<div style="background:#0A2240;color:#FFFFFF;padding:22px 26px;border-radius:12px">
<span style="color:#9CC9EE;font-size:14px;letter-spacing:2px"><b>AI FOR ACADEMIC RESEARCH SOCIETY · SESSION 6</b></span>
<h1 style="color:#FFFFFF;margin:10px 0 4px;font-family:Georgia,serif">Working with Databases</h1>
<span style="color:#C9D6E6;font-size:18px">Find, connect and question football data with SQLite</span>
</div>

Today we will open a real database, look for records, make one clearly fictional change, and ask what the numbers can (and cannot) tell us.

**A database is not an answer machine.** It stores records in an organised way. Our choices about which records to count, compare and leave out shape the answer.


## Before you begin: open the notebook and get the data

1. From the public GitHub repository, download this notebook from the [Session 6 folder](https://github.com/CharterhouseAI/AI-for-Academic-Research-Society/tree/main/session-6) and open it in JupyterLite.
2. Wait for the **Python (Pyodide)** kernel to become ready, then run the **Get the lesson database** cell below. It downloads and unpacks the lesson data beside this notebook. The compressed download is about **34 MB**; the database uses about **91.5 MB** once unpacked.
3. If the automatic download is blocked, download `transfermarkt-lesson.sqlite.gz` from that same folder, upload it into the JupyterLite File Browser beside this notebook, and run the cell again.
4. Run later cells with **Shift + Enter**. JupyterLite uses its browser workspace, not a path on your computer.

The database contains four tables: `players`, `clubs`, `player_valuations` and `transfers`. The writing exercise changes only your JupyterLite workspace copy.


In [ ]:
# Get a workspace copy of the lesson database.
import gzip
import io
import os
import shutil

DB_PATH = "transfermarkt-lesson.sqlite"
COMPRESSED_PATH = "transfermarkt-lesson.sqlite.gz"
DATABASE_URL = (
    "https://raw.githubusercontent.com/CharterhouseAI/"
    "AI-for-Academic-Research-Society/main/session-6/"
    "transfermarkt-lesson.sqlite.gz"
)

if os.path.isfile(DB_PATH):
    print("Database is ready:", DB_PATH)
elif os.path.isfile(COMPRESSED_PATH):
    print("Unpacking the uploaded database file...")
    with gzip.open(COMPRESSED_PATH, "rb") as source, open(DB_PATH, "wb") as target:
        shutil.copyfileobj(source, target)
else:
    from pyodide.http import pyfetch

    print("Downloading the lesson database from GitHub...")
    response = await pyfetch(DATABASE_URL)
    response.raise_for_status()
    compressed_data = await response.bytes()
    with gzip.GzipFile(fileobj=io.BytesIO(compressed_data), mode="rb") as source:
        with open(DB_PATH, "wb") as target:
            shutil.copyfileobj(source, target)

print(f"Ready: {os.path.getsize(DB_PATH) / (1024 * 1024):.1f} MiB")


## 1 · Explore the database

### 1.1 See the files and connect

This first cell checks that Python can see the uploaded database and shows the SQLite version included with Python. If the file is missing, check the File Browser, its exact spelling, and that it is beside this notebook.


In [ ]:
import os
import sqlite3
from contextlib import closing

# TRY EDITING: change this only if you uploaded the database into a subfolder.
DB_PATH = "transfermarkt-lesson.sqlite"

print("Notebook workspace:", os.getcwd())
print("Files Python can see:")
for file_name in os.listdir():
    print(" -", file_name)

if not os.path.isfile(DB_PATH):
    raise FileNotFoundError(
        "I cannot see " + DB_PATH +
        ". Upload it with the JupyterLite File Browser beside this notebook."
    )

print("SQLite version:", sqlite3.sqlite_version)


Now ask SQLite which tables are in the file. `connect` opens the database; `execute` sends a small SQL question; `fetchall` collects its answer. The `with closing(...)` pattern makes sure the connection is closed when we finish.


In [ ]:
with closing(sqlite3.connect(DB_PATH)) as connection:
    table_rows = connection.execute(
        "SELECT name FROM sqlite_master "
        "WHERE type = 'table' AND name NOT LIKE 'sqlite_%' "
        "ORDER BY name"
    ).fetchall()

print("Tables in this lesson database:")
for (table_name,) in table_rows:
    print(" -", table_name)


### 1.2 A table has columns and rows

A **column** describes one kind of information. A **row** is one record. `PRAGMA table_info(...)` asks SQLite about a table's columns.

**TRY EDITING:** change `players` to `clubs`, `transfers` or `player_valuations`.


In [ ]:
# TRY EDITING: choose one of the four table names.
table_to_inspect = "players"

with closing(sqlite3.connect(DB_PATH)) as connection:
    column_rows = connection.execute(
        "PRAGMA table_info(" + table_to_inspect + ")"
    ).fetchall()

print("Columns in", table_to_inspect)
for column in column_rows:
    print("-", column[1], "| SQLite type:", column[2])


Now select a few columns and a few rows. SQL reads from a named table with `SELECT ... FROM ...`. `LIMIT 5` keeps the sample small.

**TRY EDITING:** select different columns, or change `LIMIT 5` to `LIMIT 10`.


In [ ]:
# TRY EDITING: change the column names or the number of sample rows.
sample_sql = '''
SELECT player_id, name, position, current_club_name, market_value_in_eur
FROM players
LIMIT 5
'''

with closing(sqlite3.connect(DB_PATH)) as connection:
    sample_rows = connection.execute(sample_sql).fetchall()

for row in sample_rows:
    print(row)


### 1.3 Search by name

`WHERE` filters rows. `LIKE` searches text; `%` stands for any number of characters. The `?` is a **parameter placeholder**: the search text is passed separately rather than pasted into the SQL.

**TRY EDITING:** search for another player or club. Try only part of a name.


In [ ]:
# TRY EDITING: enter a player name or part of one.
player_search = "Kylian"
player_pattern = "%" + player_search + "%"

with closing(sqlite3.connect(DB_PATH)) as connection:
    player_matches = connection.execute(
        "SELECT player_id, name, current_club_name, position "
        "FROM players WHERE name LIKE ? COLLATE NOCASE "
        "ORDER BY name LIMIT 12",
        (player_pattern,)
    ).fetchall()

print("Player matches:")
for row in player_matches:
    print(row)

# TRY EDITING: try a club name or part of one.
club_search = "Arsenal"
club_pattern = "%" + club_search + "%"

with closing(sqlite3.connect(DB_PATH)) as connection:
    club_matches = connection.execute(
        "SELECT club_id, name, domestic_competition_id "
        "FROM clubs WHERE name LIKE ? COLLATE NOCASE "
        "ORDER BY name LIMIT 12",
        (club_pattern,)
    ).fetchall()

print("\nClub matches:")
for row in club_matches:
    print(row)


### 1.4 A fuzzy search can help with a typo

An exact text search may miss a name if a letter or accent is different. Python's built-in `difflib` can suggest strings that look similar.

**TRY EDITING:** misspell a player name on purpose. Then change `players` to `clubs` and try a club name. This is character similarity—not a search for meaning—so check the suggestions yourself.


In [ ]:
import difflib
from contextlib import closing

# TRY EDITING: try a spelling with a missing or changed letter.
search_text = "Kylian Mbape"

# TRY EDITING: choose "players" or "clubs".
search_table = "players"

if search_table == "players":
    sql_for_names = "SELECT name FROM players WHERE name IS NOT NULL"
elif search_table == "clubs":
    sql_for_names = "SELECT name FROM clubs WHERE name IS NOT NULL"
else:
    raise ValueError("Choose only 'players' or 'clubs'.")

with closing(sqlite3.connect(DB_PATH)) as connection:
    original_names = [row[0] for row in connection.execute(sql_for_names)]

# Compare case-insensitively while keeping the original spelling for display.
name_lookup = {name.casefold(): name for name in original_names}
suggestions = difflib.get_close_matches(
    search_text.casefold(),
    list(name_lookup.keys()),
    n=5,
    cutoff=0.45
)

print("Closest spelling matches:")
for suggestion in suggestions:
    print("-", name_lookup[suggestion])


**Pause and think:** Did the fuzzy search find the name you meant? When might a similar-looking name be the wrong person or club?

`LIKE` and fuzzy matching find words by their characters. Neither search understands football context or proves that two names refer to the same entity.


## 2 · Write one fictional player record

So far we have only read information. Now we will insert one clearly fictional row into the `players` table, then query it back.

The code chooses a new ID and checks for the demo name before inserting, so running it again will not add another copy. This changes your JupyterLite workspace copy, not the teacher's original file.


In [ ]:
fictional_name = "Alex Example (FICTIONAL)"

with closing(sqlite3.connect(DB_PATH)) as connection:
    existing = connection.execute(
        "SELECT player_id, name, position FROM players WHERE name = ?",
        (fictional_name,)
    ).fetchone()

    if existing is None:
        new_player_id = connection.execute(
            "SELECT COALESCE(MAX(player_id), 0) + 1 FROM players"
        ).fetchone()[0]

        # A fictional example: only the columns we supply receive values.
        with connection:
            connection.execute(
                "INSERT INTO players "
                "(player_id, first_name, last_name, name, position, sub_position) "
                "VALUES (?, ?, ?, ?, ?, ?)",
                (
                    new_player_id,
                    "Alex",
                    "Example",
                    fictional_name,
                    "Midfield",
                    "Central Midfield"
                )
            )
        print("Added one fictional practice row.")
    else:
        print("The fictional practice row is already in this workspace copy.")

    added_row = connection.execute(
        "SELECT player_id, name, position, sub_position "
        "FROM players WHERE name = ?",
        (fictional_name,)
    ).fetchone()

print("Sample row:", added_row)


**Your turn:** change the fictional player's name and position, then run the cell again. What needs to change so that the new name is inserted only once?

**Optional reset:** after you have seen the row, run this cell to remove only the fictional example.


In [ ]:
fictional_name = "Alex Example (FICTIONAL)"

with closing(sqlite3.connect(DB_PATH)) as connection:
    with connection:
        cursor = connection.execute(
            "DELETE FROM players WHERE name = ?",
            (fictional_name,)
        )

print("Fictional rows removed:", cursor.rowcount)


## 3 · Ask a larger question

### 3.1 Which clubs have the largest recorded transfer spend?

We will group positive, recorded fees by the **destination** club and add them. Transfer fees are nominal euros, not adjusted for inflation.

There are many missing or zero fees in this file: **61,526 are NULL, 96,085 are zero, and 17,554 are positive**. A zero is not proof that a transfer was free; it can also mean that no fee was recorded. This example counts only positive fees.

The transfer table includes dates as late as **2030-06-30**. The fixed end date below avoids counting future-dated records in this lesson. Change the dates to ask a different question.


In [ ]:
import sqlite3
from contextlib import closing

# TRY EDITING: compare a different date window.
start_date = "1993-07-01"
end_date = "2026-11-18"

spend_sql = '''
SELECT to_club_id, to_club_name,
       SUM(transfer_fee) AS recorded_spend_eur,
       COUNT(*) AS positive_fee_transfers
FROM transfers
WHERE transfer_fee > 0
  AND transfer_date BETWEEN ? AND ?
GROUP BY to_club_id, to_club_name
ORDER BY recorded_spend_eur DESC
LIMIT 10
'''

with closing(sqlite3.connect(DB_PATH)) as connection:
    top_spending_clubs = connection.execute(
        spend_sql, (start_date, end_date)
    ).fetchall()

print("Top destination clubs by recorded positive fees")
print("Period:", start_date, "to", end_date)
for rank, row in enumerate(top_spending_clubs, start=1):
    club_id, club_name, total_eur, number_of_transfers = row
    print(
        f"{rank:>2}. {club_name or 'Unknown club'} "
        f"(club {club_id}) — €{total_eur:,.0f} "
        f"across {number_of_transfers} positive-fee records"
    )


A bar chart makes the ranking easier to compare. The chart shows **recorded positive fees only**; it does not show unreported or zero-fee transfers.


In [ ]:
if top_spending_clubs:
    labels = [row[1] or "Unknown club" for row in top_spending_clubs]
    totals_billions = [row[2] / 1_000_000_000 for row in top_spending_clubs]

    try:
        import matplotlib.pyplot as plt
    except ModuleNotFoundError as error:
        if error.name != "matplotlib":
            raise
        # A visible fallback keeps the example usable in kernels without Matplotlib.
        print("Matplotlib is not available here; showing a simple text bar chart.")
        largest_total = max(totals_billions)
        for label, total in zip(labels, totals_billions):
            bar_length = round(total / largest_total * 32)
            print(f"{label:18} {'#' * bar_length} €{total:.2f}bn")
    else:
        plt.figure(figsize=(9, 5))
        plt.barh(labels[::-1], totals_billions[::-1], color="#274264")
        plt.xlabel("Recorded positive transfer fees (€ billions)")
        plt.title("Top 10 destination clubs by recorded transfer spend")
        plt.grid(axis="x", alpha=0.25)
        plt.tight_layout()
        plt.show()
else:
    print("No positive-fee transfers were found in this date range.")


**Interpret carefully:** this is a sum of recorded fees in the selected years. It is not a complete account of club budgets. Fees are not inflation-adjusted, and the many missing or zero values make the ranking incomplete.


### 3.2 The five highest current player market values

Here **most valuable** means the five largest values in `players.market_value_in_eur` in this database snapshot. We calculate each player's age on the day you run the notebook.

For comparison, we find each player's **highest positive recorded transfer fee** and calculate their age on that transfer date. This is one practical definition of “most valuable transfer”; it is not the same as highest market value.


In [ ]:
import sqlite3
from contextlib import closing
from datetime import date

def age_on(birth_text, event_text):
    """Return a whole-year age on a date, or None if either date is missing."""
    if not birth_text or not event_text:
        return None

    birth_date = date.fromisoformat(str(birth_text)[:10])
    event_date = date.fromisoformat(str(event_text)[:10])
    birthday_not_reached = (event_date.month, event_date.day) < (
        birth_date.month, birth_date.day
    )
    return event_date.year - birth_date.year - int(birthday_not_reached)

as_of = date.today()

with closing(sqlite3.connect(DB_PATH)) as connection:
    top_players = connection.execute(
        "SELECT player_id, name, date_of_birth, market_value_in_eur "
        "FROM players "
        "WHERE market_value_in_eur IS NOT NULL "
        "  AND date_of_birth IS NOT NULL "
        "ORDER BY market_value_in_eur DESC, name "
        "LIMIT 5"
    ).fetchall()

    print("Current age is calculated as of", as_of.isoformat())
    for player_id, name, birth_date, current_value in top_players:
        current_age = age_on(birth_date, as_of.isoformat())
        most_valuable_transfer = connection.execute(
            "SELECT transfer_date, transfer_fee, from_club_name, to_club_name "
            "FROM transfers "
            "WHERE player_id = ? "
            "  AND transfer_fee > 0 "
            "  AND transfer_date IS NOT NULL "
            "ORDER BY transfer_fee DESC, transfer_date ASC "
            "LIMIT 1",
            (player_id,)
        ).fetchone()

        print(
            f"\n{name}: profile market value €{current_value:,.0f}; "
            f"age {current_age}"
        )
        if most_valuable_transfer is None:
            print("  No positive, dated transfer fee is recorded.")
        else:
            transfer_date, fee, from_club, to_club = most_valuable_transfer
            age_at_transfer = age_on(birth_date, transfer_date)
            print(
                f"  Highest positive recorded fee: €{fee:,.0f} "
                f"({from_club} → {to_club}, {transfer_date}); "
                f"age then: {age_at_transfer}"
            )


The player profile's market value is a snapshot from this dataset, not necessarily today's live value. Some players have no positive transfer fee recorded. “No recorded paid transfer” does **not** prove that the player never moved clubs.

**Pause and think:** What did the ranking leave out? Would an age pattern prove that age causes a higher or lower fee?


## 4 · Ask an AI tool to help with a new question

You have practised opening the file, searching, reading, writing, joining ideas through IDs, and making a chart. Now choose a question that interests you.

Run the cell below and copy the **whole printed prompt** into Microsoft Copilot. Copilot should ask you what you want to investigate, help narrow the idea, and return one complete cell. Paste it into a new notebook cell and run it.

If there is an error, give Copilot the **complete error message and the complete code cell**. Ask it to explain the problem and return a corrected, self-contained cell.


In [ ]:
copilot_context = r'''
I am a beginner exploring a football-transfer database in a JupyterLite notebook for
the AI for Academic Research Society. Please start by asking what I would like to
find out. Ask one clear question and wait for my answer. Then help me turn my idea
into a small, answerable data question. Explain the measure and its limitations.
Return exactly one complete, self-contained, commented Python code cell that I can
paste into a new cell and run.

ENVIRONMENT
- This is JupyterLite with the Python (Pyodide) kernel, running in a web browser.
- I am using a desktop computer, but Python can only read files uploaded into the
  JupyterLite File Browser workspace.
- The database file is named exactly: transfermarkt-lesson.sqlite
- The notebook and database are in the same workspace folder. Use the relative
  path "transfermarkt-lesson.sqlite"; never use a Windows, network, or computer path.
- Python's standard library is available, including sqlite3, datetime, difflib,
  math, statistics, and csv. SQLite is built in; do not install packages.
- Matplotlib is available in the Pyodide package collection for basic charts.
  Use matplotlib.pyplot when a graph would help; do not use pip or install packages.
- Do not use network access, subprocesses, shell commands, sockets, threading,
  multiprocessing, or files outside the JupyterLite workspace.

DATABASE
- This SQLite database is about 91.5 MB and contains exactly four tables.
- It has about 932,411 rows total. Values may be NULL. The tables were copied
  from a larger dataset; not every record has a complete set of values.
- Use sqlite3.connect("transfermarkt-lesson.sqlite") and always close connections.
- Use SQL parameter placeholders (?) for values. Do not assemble SQL by inserting
  a learner's search text directly into the query.
- Analyze only the tables and exact columns listed below. IDs can connect related
  rows, but the file does not enforce foreign-key constraints. Inspect SQLite types:
  club IDs are TEXT in clubs and players.current_club_id, but INTEGER in some
  other tables. Cast explicitly if a join needs consistent types.

TABLE: players — 50,149 player profile rows
Typical content: player identity, biography, position, current-club details, and
market-value snapshot. Some biography and current-value fields are blank.
Exact columns:
- player_id: Transfermarkt player identifier.
- first_name: given name.
- last_name: family name.
- name: displayed full name.
- last_season: latest season label in the profile.
- current_club_id: identifier of the player's current club, stored as text.
- player_code: URL-friendly player slug.
- country_of_birth: birth country.
- city_of_birth: birth city.
- country_of_citizenship: citizenship.
- date_of_birth: birth date; 49 rows have no date.
- sub_position: specific playing role, such as Centre-Forward.
- position: broad position group, such as Attack or Midfield.
- foot: preferred foot.
- height_in_cm: height in centimetres.
- contract_expiration_date: recorded contract end date.
- agent_name: player's agent or agency.
- image_url: portrait-image address.
- international_caps: recorded national-team appearances.
- international_goals: recorded national-team goals.
- current_national_team_id: national-team identifier.
- url: player profile address.
- current_club_domestic_competition_id: current club competition code.
- current_club_name: current club name.
- market_value_in_eur: market-value snapshot in euros; 8,621 rows are blank.
- highest_market_value_in_eur: highest recorded profile market value in euros;
  8,621 rows are blank.

TABLE: clubs — 796 club profile rows
Typical content: club identifiers, competition, squad summary, stadium and profile
metadata. total_market_value is blank in all 796 rows in this copy.
Exact columns:
- club_id: club identifier, stored as text.
- club_code: URL-friendly club slug.
- name: club display name.
- domestic_competition_id: competition code.
- total_market_value: squad/club value field; blank in this copy.
- squad_size: squad-size count.
- average_age: average squad age.
- foreigners_number: count of foreign players.
- foreigners_percentage: foreign-player percentage.
- national_team_players: count of players with national-team representation.
- stadium_name: stadium name.
- stadium_seats: stadium capacity.
- net_transfer_record: formatted text summary, not a numeric fee column.
- coach_name: coach name.
- last_season: latest season label in this profile.
- filename: source data filename/path text.
- url: club profile address.

TABLE: player_valuations — 656,301 valuation-history rows
Typical content: a player's market value on a date and club/competition labels.
Dates range from 2000-01-20 to 2026-06-12.
Exact columns:
- player_id: player identifier; links to players.player_id.
- date: date of this valuation observation.
- market_value_in_eur: estimated player value in euros on that date.
- current_club_name: club name recorded for the observation.
- current_club_id: club identifier for the observation, stored as an integer.
- player_club_domestic_competition_id: domestic competition code; some rows are blank.

TABLE: transfers — 175,165 transfer rows
Typical content: player, transfer date/season, origin and destination clubs, fee,
and market value around the transfer. Dates range from 1993-07-01 to 2030-06-30;
some records are future-dated relative to the lesson.
Exact columns:
- player_id: player identifier; links to players.player_id.
- transfer_date: recorded transfer date.
- transfer_season: season label such as 25/26.
- from_club_id: origin club identifier, stored as an integer.
- to_club_id: destination club identifier, stored as an integer.
- from_club_name: origin club name.
- to_club_name: destination club name.
- transfer_fee: recorded transfer fee in euros; 61,526 are NULL, 96,085 are zero,
  and only 17,554 are positive. Zero may mean free, unreported, or otherwise
  recorded as zero; do not assume all zeroes are free transfers.
- market_value_in_eur: recorded player market value around the transfer in euros;
  some rows are blank.
- player_name: player name in the transfer row.

ANALYSIS AND CODE RULES
- Make a graph whenever a graph would help answer my question. Use a suitable
  matplotlib chart with a title, labelled axes, readable labels, and units. For
  very large datasets, aggregate in SQL and plot only the summarized result.
- Include all imports, database connection, SQL, and helper functions needed in
  the one cell. Do not rely on earlier cells or variables.
- Use LIMIT for example rows. Use clear variable names and short comments.
- For dates, distinguish the dataset's recorded dates from the date I run the code.
  If a question concerns historical transfers, let me choose explicit start/end
  dates and mention future-dated rows.
- State clearly which rows were included or excluded. Treat missing values as
  missing, not as zero. In transfer-fee analysis, distinguish NULL, zero, and
  positive recorded fees.
- Use market values and transfer fees as different measures. Fees are nominal
  euros and are not adjusted for inflation in this database.
- A graph or correlation can suggest a pattern; it does not by itself prove a
  cause or explain what a player, club, or transfer means.
- Do not silently hide errors or invent replacement data. If a file or column is
  missing, say so and show me how to check the uploaded file and exact schema.

Once I tell you my question, help me make it precise, then return the one complete
code cell and briefly tell me what to look for in its output.
'''

print(copilot_context)


### Our reminder

Search results, rankings and charts are **measurements**, not explanations. Check which records were included, what is missing, and whether the pattern supports the interpretation you want to make.

**Next step:** choose one question of your own, then use the Copilot prompt above to build and test a small analysis.
